# Uni-Projekt: Large Language Models from Scratch
**Gruppe:** Phillip, Konstantin, Fabian

**Ziel:** Aufbau und Training eines eigenen LLM-Modells, mithilfe eines Datensatzes der Kochrezepte enthält.

# Datensatz einlesen
* Den CSV Datensatz einlesen
* Die ersten Zeilen ausgeben

In [1]:
import os
import pandas as pd
from sympy import root

root_dir = file_path = os.path.join("C:\\", "resources", "Uni", "LLMs", "LLMs-from-scratch")
file_path = file_path = os.path.join(root_dir, "datasets", "reduced_dataset_100k.csv")

if os.path.exists(file_path):
    df = pd.read_csv(file_path)
    print(df.info())
    print(df.head())   
else:    print(f"File '{file_path}' does not exist.")



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 7 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   Unnamed: 0   100000 non-null  int64 
 1   title        100000 non-null  object
 2   ingredients  100000 non-null  object
 3   directions   100000 non-null  object
 4   link         100000 non-null  object
 5   source       100000 non-null  object
 6   NER          100000 non-null  object
dtypes: int64(1), object(6)
memory usage: 5.3+ MB
None
   Unnamed: 0                  title  \
0           0    No-Bake Nut Cookies   
1           1  Jewell Ball'S Chicken   
2           2            Creamy Corn   
3           3          Chicken Funny   
4           4   Reeses Cups(Candy)     

                                         ingredients  \
0  ["1 c. firmly packed brown sugar", "1/2 c. eva...   
1  ["1 small jar chipped beef, cut up", "4 boned ...   
2  ["2 (16 oz.) pkg. frozen corn", "1 (8 oz.) pkg

- Die CSV-Datei als einen langen String formatieren

In [2]:
def format_csv(row):
    title = str(row['title'])
    ingredients = str(row['ingredients']).replace('[', '').replace(']', '').replace("'", "").replace('"', '')
    directions = str(row['directions']).replace('[', '').replace(']', '').replace("'", "").replace('"', '')
    return f"Recepie: {title}\nIngredients:{ingredients}\nDirections:{directions}"

raw_text = "".join(df.apply(format_csv, axis=1))
print(f"Der gesamte Datensatz wurde in einen String umgewandelt.")
print(f"Gesamtanzahl der Zeichen: {len(raw_text)}")
print("\nVorschau der ersten 300 Zeichen:")
print(raw_text[:300])

    

Der gesamte Datensatz wurde in einen String umgewandelt.
Gesamtanzahl der Zeichen: 46874254

Vorschau der ersten 300 Zeichen:
Recepie: No-Bake Nut Cookies
Ingredients:1 c. firmly packed brown sugar, 1/2 c. evaporated milk, 1/2 tsp. vanilla, 1/2 c. broken nuts (pecans), 2 Tbsp. butter or margarine, 3 1/2 c. bite size shredded rice biscuits
Directions:In a heavy 2-quart saucepan, mix brown sugar, nuts, evaporated milk and bu


- Text kürzen

# Tokenizer bauen


- Tokenizer auf mit einfachen Textbeispielen bauen und später auf den Datensatz anwenden
- Mithilfe von RegEx die Wörter trennen

In [3]:
import re

text = "This is a sample text: with numbers 123, special characters !@# and a few more.  ."
def split_text(text):
    tokens = re.split(r'([,.:;?_!"()\']|--|\s)', text)
    tokens = [token for token in tokens if token.strip()]
    return tokens


result = split_text(text)


print(result)

['This', 'is', 'a', 'sample', 'text', ':', 'with', 'numbers', '123', ',', 'special', 'characters', '!', '@#', 'and', 'a', 'few', 'more', '.', '.']


- Trennung der Wörter auf den Datensatz anwenden

In [4]:
split = split_text(raw_text)
print(f"Anzahl der Wörter: {len(split)}")
print(split[:20])

Anzahl der Wörter: 11037578
['Recepie', ':', 'No-Bake', 'Nut', 'Cookies', 'Ingredients', ':', '1', 'c', '.', 'firmly', 'packed', 'brown', 'sugar', ',', '1/2', 'c', '.', 'evaporated', 'milk']


&nbsp;
## 2.3 Converting tokens into token IDs

- Vokabular das alle Wort-Tokens im Datensatzenthält

In [5]:
all_words = sorted(set(split))
vocab_size = len(all_words)
print(vocab_size)

29679


In [6]:
vocab = {token:integer for integer,token in enumerate(all_words)}

In [7]:
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

('!', 0)
('"', 1)
('#', 2)
('#1', 3)
('#10', 4)
('#2', 5)
('#202', 6)
('#28', 7)
('#3', 8)
('#303', 9)
('#4', 10)
('#40', 11)
('#42', 12)
('#48', 13)
('#5', 14)
('#6', 15)
('#7', 16)
('#A', 17)
('$$$', 18)
('$100', 19)
('$15', 20)
('$20', 21)
('$3', 22)
('$4', 23)
('$5', 24)
('%', 25)
('&', 26)
('&/or', 27)
("'", 28)
('(', 29)
(')', 30)
('*', 31)
('**', 32)
('****', 33)
('**Christmas', 34)
('**I', 35)
('**NOTE', 36)
('**The', 37)
('**two', 38)
('*1', 39)
('*2', 40)
('*50-50', 41)
('*6', 42)
('*Add', 43)
('*All', 44)
('*Alternate', 45)
('*Always', 46)
('*Amount', 47)
('*Andouille', 48)
('*Angel', 49)
('*Any', 50)


In [8]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
                                
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [16]:
tokenizer = SimpleTokenizerV1(vocab)

text = """"Recepie: No-Bake Nut Cookies Ingredients: 1c. firmly packed brown sugar. Soft baked."""
ids = tokenizer.encode(text)
print(ids)

[1, 12388, 1694, 10552, 10666, 4715, 8026, 1694, 809, 289, 21013, 24646, 18224, 27813, 289, 13816, 17604, 289]


In [17]:
tokenizer.decode(ids)

'" Recepie : No-Bake Nut Cookies Ingredients : 1c. firmly packed brown sugar. Soft baked.'

In [18]:
tokenizer.decode(tokenizer.encode(text))

'" Recepie : No-Bake Nut Cookies Ingredients : 1c. firmly packed brown sugar. Soft baked.'

# Add specials tokens
- unknown
- end of text
- GPT2 semantic

In [20]:
all_tokens = sorted(list(set(split)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token:integer for integer,token in enumerate(all_tokens)}

In [21]:
#adjusted tokenizer
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [
            item if item in self.str_to_int 
            else "<|unk|>" for item in preprocessed
        ]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

In [22]:
tokenizer = SimpleTokenizerV2(vocab)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."

text = " <|endoftext|> ".join((text1, text2))

print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [23]:
tokenizer.encode(text)

[7572,
 250,
 20271,
 29629,
 23051,
 28106,
 1699,
 29679,
 7978,
 28219,
 29680,
 29680,
 24265,
 28219,
 24673,
 289]

In [24]:
tokenizer.decode(tokenizer.encode(text))

'Hello, do you like tea? <|endoftext|> In the <|unk|> <|unk|> of the palace.'

# Byte Pair Encoding

Usage of a an already implemented byte pair encoding from open ai for further development of the LLM. The previous tokenizer was just for training purposes.

In [25]:
import importlib
import tiktoken

print("tiktoken version:", importlib.metadata.version("tiktoken"))

tiktoken version: 0.9.0


In [27]:
tokenizer = tiktoken.get_encoding("cl100k_base")

In [28]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print(integers)

[9906, 11, 656, 499, 1093, 15600, 30, 220, 100257, 763, 279, 7160, 32735, 7317, 2492, 1073, 1063, 16476, 17826, 13]


In [29]:
strings = tokenizer.decode(integers)

print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.
